# Wind Power Generation Forecasting

## Project Overview

Forecasts **daily wind power generation** (MWh) over a 14-day horizon. Synthetic data spans 2 years with seasonal patterns (windier winters), high daily variability, and capacity growth.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily wind generation, predict the next 14 days. Wind forecasting is essential for grid balancing, market bidding, and managing conventional generation reserves.

## Why This Project Matters

Wind is the most variable renewable source. Forecast errors require expensive backup generation. As wind penetration grows globally, forecasting accuracy directly impacts electricity costs and grid reliability.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily wind generation
- Seasonal pattern (higher in winter, lower in summer)
- Very high day-to-day variability (weather-driven)
- Slight upward trend (capacity additions)
- No weekly pattern

| Column | Description |
|--------|-------------|
| `date` | Date |
| `wind_mwh` | Daily wind energy generation (MWh) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'wind_mwh'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Seasonal: windier winter, calmer summer (opposite of solar)
seasonal = 400 + 150 * np.cos(2 * np.pi * (t - 15) / 365)
trend = 0.15 * t

# Wind is highly variable — use Weibull-like noise
wind_var = np.random.weibull(2.5, N_POINTS) * 200
noise = np.random.normal(0, 50, N_POINTS)

wind_mwh = seasonal + trend + wind_var + noise
wind_mwh = np.maximum(wind_mwh, 5).round(1)

df = pd.DataFrame({'date': dates, 'wind_mwh': wind_mwh})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,wind_mwh
0,2022-01-01,712.5
1,2022-01-02,835.6
2,2022-01-03,784.3
3,2022-01-04,843.9
4,2022-01-05,689.8


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('wind_mwh Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("wind_mwh Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

wind_mwh Statistics:
count     730.000000
mean      634.714110
std       140.035974
min       291.000000
25%       528.825000
50%       640.550000
75%       728.900000
max      1033.500000
Name: wind_mwh, dtype: float64

CV: 0.221
Skewness: 0.073


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      165.8 | RMSE:      192.4 | MAPE: 18.61%
Seasonal Naive (7)             | MAE:      102.8 | RMSE:      116.7 | MAPE: 12.36%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                          
KernelRidge                      1051.850388 -79.834645  636.720882    0.037670
GaussianProcessRegressor          920.999581 -69.769199  595.761540    0.089291
MLPRegressor                      697.999650 -52.615358  518.555122    0.428075
DummyRegressor                    100.768074  -6.674467  196.188905    0.008834
QuantileRegressor                  96.886122  -6.375856  192.334205    0.061707
NuSVR                              80.376618  -5.105894  174.994719    0.030604
SVR                                71.016428  -4.385879  164.353374    0.028681
DecisionTreeRegressor              70.413323  -4.339486  163.643993    0.017311
LinearSVR                          68.051982  -4.157845  160.836442    0.011337
XGBRegressor                       41.472679  -2.113283  124.956722    0.497754


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:       68.4 | RMSE:       87.5 | MAPE:  8.06%
FLAML Test (extra_tree)        | MAE:      108.7 | RMSE:      126.6 | MAPE: 12.35%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       92.9 | RMSE:      104.3 | MAPE: 10.99%
SF AutoETS                     | MAE:       90.4 | RMSE:      103.4 | MAPE: 11.11%
SF SeasonalNaive               | MAE:       89.2 | RMSE:      107.5 | MAPE: 10.77%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model        MAE       RMSE      MAPE
     FLAML (extra_tree)  68.433966  87.504942  8.057176
       SF SeasonalNaive  89.200000 107.469152 10.768437
             SF AutoETS  90.439985 103.394555 11.109541
           SF AutoARIMA  92.874408 104.284601 10.991956
     Seasonal Naive (7) 102.771429 116.731695 12.363458
FLAML Test (extra_tree) 108.670371 126.625804 12.354144
     Naive (Last Value) 165.757143 192.446972 18.614051

Top 3:
             model       MAE       RMSE      MAPE
FLAML (extra_tree) 68.433966  87.504942  8.057176
  SF SeasonalNaive 89.200000 107.469152 10.768437
        SF AutoETS 90.439985 103.394555 11.109541


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 79.87, Std: 98.26


## Interpretation and Business Insight

- Wind generation is seasonal (winter > summer) — opposite to solar
- Day-to-day variability is very high (weather-driven)
- Wind forecast errors are larger than solar errors
- Capacity growth adds an upward trend
- Lag features have limited power due to low autocorrelation

## Limitations

1. Synthetic — real wind depends on wind speed, direction, turbine specs
2. No weather/wind speed features (key driver)
3. No hourly profile (wind varies significantly intraday)
4. Single wind farm — real grids aggregate across regions

## How to Improve This Project

1. Add wind speed and direction forecasts
2. Use hourly data for dispatch planning
3. Model power curve (wind speed → generation) explicitly
4. Aggregate across multiple wind farms for smoother forecasts
5. Use NWP (numerical weather prediction) model output

## Production Considerations

- Day-ahead and intraday forecasts for market bidding
- Integration with NWP weather models
- Ensemble forecasting for uncertainty quantification
- Reserve requirement optimization

## Common Mistakes

1. Expecting smooth patterns — wind is inherently noisy
2. Not using weather forecasts as primary input
3. Over-fitting on training data with low autocorrelation
4. Ignoring the power curve (wind speed to generation is non-linear)

## Mini Challenge / Exercises

1. Add a synthetic wind speed feature and model the power curve
2. Compare daily vs weekly aggregation accuracy
3. Build a probabilistic forecast with quantile regression
4. Combine solar and wind forecasts for total renewable prediction

## Final Summary / Key Takeaways

1. Wind forecasting is the most challenging renewable forecasting task
2. High variability limits pure time-series model accuracy
3. Weather inputs (wind speed) are essential for real deployment
4. Seasonal patterns (winter > summer) provide a baseline
5. Aggregation across farms reduces forecast error significantly

In [18]:
import json
metrics = {
    'project': 'Wind Power Generation Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Wind Power Generation Forecasting — Complete ===")

Metrics saved.

=== Wind Power Generation Forecasting — Complete ===
